# 01 — Data Collection

This notebook walks through fetching food price data from **FAO FAOSTAT** and **WFP/HDX** for Algeria.

**Steps:**
1. Load configuration
2. Fetch FAO producer price data
3. Fetch WFP market price data
4. Validate and preview raw data
5. Save to Parquet

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.utils.helpers import load_config, setup_logging
from src.data_ingestion.fao_connector import FAOConnector
from src.data_ingestion.wfp_connector import WFPConnector

config = load_config('../config/config.yaml')
setup_logging(config.get('logging', {}))
print('Configuration loaded ✅')

In [ ]:
# ── FAO Data ──────────────────────────────────────────────────
fao = FAOConnector(config=config)
fao_df = fao.fetch_price_data(year_start=2015, year_end=2024)
print(f'FAO records: {len(fao_df):,}')
fao_df.head()

In [ ]:
print('FAO data types:')
print(fao_df.dtypes)
print('\nMissing values:')
print(fao_df.isna().sum())
print('\nYear range:', fao_df['year'].min(), '–', fao_df['year'].max())

In [ ]:
# ── WFP Data ──────────────────────────────────────────────────
wfp = WFPConnector(config=config)
wfp_df = wfp.fetch_price_data()
print(f'WFP records: {len(wfp_df):,}')
wfp_df.head()

In [ ]:
print('WFP products:', wfp_df['product'].unique())
print('WFP regions:', wfp_df['region'].unique())
print('\nDate range:', wfp_df['date'].min(), '→', wfp_df['date'].max())

In [ ]:
# ── Quick Plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# WFP: price distribution
wfp_df.groupby('product')['price'].median().sort_values().plot(
    kind='barh', ax=axes[0], color='#006233', alpha=0.85
)
axes[0].set_title('Median WFP Price by Product (DZD)')
axes[0].set_xlabel('DZD')

# Records per region
wfp_df.groupby('region').size().plot(
    kind='bar', ax=axes[1], color='#1565c0', alpha=0.85, rot=30
)
axes[1].set_title('WFP Observations by Region')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# ── Save raw data ─────────────────────────────────────────────
fao_path = fao.fetch_and_save(year_start=2015, year_end=2024)
wfp_path = wfp.fetch_and_save()
print(f'Saved FAO → {fao_path}')
print(f'Saved WFP → {wfp_path}')